<a href="https://colab.research.google.com/github/matrixportalx/Sd-1.5-Converting-to-Qualcomm-QNN-Model/blob/claude%2Fqnn-model-conversion-snapdragon7-rsk8og/SD15_NPU_Official_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SD 1.5 → Qualcomm NPU (Local Dream / Ruya) — RESMİ HAT

Bu defter, Local Dream'in **kendi dönüştürme scriptlerini** (`npuconvertv2`)
QNN SDK **2.28** ile koşar. Rehber: `ld-guide.chino.icu/conversion/sd15`

**Neden bu hat:** kendi yazdığımız hat cihazda yüklenmeyen paketler üretiyordu.
Sebepleri resmi scriptlerde görüldü:

| bizim (eski) | resmi |
|---|---|
| `qairt-converter` → DLC | `qnn-onnx-converter` → `model.cpp` → `.so` |
| `--act_bitwidth 8` + io16 hilesi | **`--act_bitwidth 16`** |
| per-channel kapalı | `--use_per_channel_quantization` |
| stok diffusers | `redefined_modules/` (MHA→SHA, Linear→Conv) |
| VTCM ayarsız | `"vtcm_mb": 2` |

**Çalışma sırası:** 1 → 2 → 3 → 4 → 5

> **GPU notu:** GPU yalnızca kalibrasyon verisi üretimini (`prepare_data.py`)
> hızlandırır — ~35 dk yerine ~3 dk. Kuantizasyon, model-lib ve context-binary
> adımları **tamamen CPU**'dur ve GPU'dan etkilenmez. Yine de yüksek RAM
> gerektiği için (rehber: ~20 GB) yüksek bellekli çalışma zamanı şart.


## 1) Ayarlar

In [ ]:
#@title Ayarlar { display-mode: "form" }
#@markdown Model dosyasının **doğrudan indirme** bağlantısı (.safetensors)
SAFETENSORS_URL = "https://civitai.com/api/download/models/3077720?fileId=2956906"  #@param {type:"string"}
MODEL_NAME = "DeliberateCyber"  #@param {type:"string"}
#@markdown Çip katmanı — `min` = Hexagon V68+ (Snapdragon 7 Gen 1 dahil)
SOC = "min"  #@param ["min", "8gen1", "8gen2"]
#@markdown `clip_skip`: modelin eğitildiği değer. Anime çoğunlukla 2.
CLIP_SKIP = 2  #@param [1, 2] {type:"raw"}
#@markdown Foto-gerçekçi model ise işaretleyin (kalibrasyon promptlarını değiştirir)
REALISTIC = True  #@param {type:"boolean"}
#@markdown Kuantizasyon örnek sayısı. Resmi tarif 400 (saatler).
#@markdown `24` = boru hattını doğrula, `150` = iyi denge, `0` = tam (400)
CALIB_LIMIT = 24  #@param {type:"integer"}
#@markdown GPU varsa CUDA torch kur (prepare_data ~10x hızlanır)
CUDA_TORCH = True  #@param {type:"boolean"}

import os
os.environ.update(
    MODEL_NAME=MODEL_NAME, SOC=SOC,
    CLIP_SKIP=str(CLIP_SKIP),
    REALISTIC="1" if REALISTIC else "0",
    CALIB_LIMIT=str(CALIB_LIMIT),
    CUDA_TORCH="1" if CUDA_TORCH else "0",
    SAFETENSORS_URL=SAFETENSORS_URL,
)
print(f"{MODEL_NAME} | soc={SOC} clip_skip={CLIP_SKIP} realistic={REALISTIC}")
print(f"calib_limit={CALIB_LIMIT} cuda_torch={CUDA_TORCH}")
!nvidia-smi -L || echo "GPU yok — prepare_data yavas olacak"


DeliberateCyber | soc=min clip_skip=2 realistic=True
calib_limit=24 cuda_torch=True
GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-3ab05dcf-8b25-329e-8119-023e2f58bba0)


## 2) Depo + araçlar

Depoyu klonlar/günceller ve `uv`'yi kurar. Çalışma zamanı sıfırlansa bile
tekrar çalıştırmak yeterli.

In [ ]:
%cd /content
REPO = "https://github.com/matrixportalx/Sd-1.5-Converting-to-Qualcomm-QNN-Model"
BRANCH = "claude/qnn-model-conversion-snapdragon7-rsk8og"
import os, subprocess
if not os.path.isdir("/content/sd-qnn/.git"):
    !git clone -b {BRANCH} {REPO} /content/sd-qnn
else:
    !cd /content/sd-qnn && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
%cd /content/sd-qnn
!pip install -q uv
!git log --oneline -1


/content
Cloning into '/content/sd-qnn'...
remote: Enumerating objects: 616, done.
remote: Counting objects: 100% (313/313), done.
remote: Compressing objects: 100% (224/224), done.
remote: Total 616 (delta 240), reused 159 (delta 89), pack-reused 303 (from 1)
Receiving objects: 100% (616/616), 881.94 KiB | 18.76 MiB/s, done.
Resolving deltas: 100% (421/421), done.
/content/sd-qnn
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.0/22.0 MB 115.2 MB/s eta 0:00:00
96e0129 (HEAD -> claude/qnn-model-conversion-snapdragon7-rsk8og, origin/claude/qnn-model-conversion-snapdragon7-rsk8og, origin/HEAD) Resmi hat: clang++ bagimliligini kur (qnn-model-lib-generator)


## 3) QNN SDK 2.28

~2 GB. **Sürüm önemli** — rehber 2.28 şart koşuyor. İndirme koparsa bu hücreyi
tekrar çalıştırın, kaldığı yerden devam eder.

In [ ]:
%cd /content/sd-qnn
import os
URL = ("https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/"
       "qualcomm_neural_processing_sdk/v2.28.0.241029.zip")
out = !python3 scripts/setup_qnn_sdk.py --dest /content/qairt --asset-url "{URL}"
print("\n".join(out[-25:]))
root = [l.split("=",1)[1] for l in out if l.startswith("QNN_SDK_ROOT=")]
assert root, "QNN_SDK_ROOT bulunamadi — yukaridaki ciktiya bakin"
os.environ["QNN_SDK_ROOT"] = root[-1].strip()
print("\nQNN_SDK_ROOT =", os.environ["QNN_SDK_ROOT"])


/content/sd-qnn
    935/954 MB (97%)
    936/954 MB (98%)
    937/954 MB (98%)
    938/954 MB (98%)
    939/954 MB (98%)
    940/954 MB (98%)
    941/954 MB (98%)
    942/954 MB (98%)
    943/954 MB (98%)
    944/954 MB (98%)
    945/954 MB (98%)
    946/954 MB (99%)
    947/954 MB (99%)
    948/954 MB (99%)
    949/954 MB (99%)
    950/954 MB (99%)
    951/954 MB (99%)
    952/954 MB (99%)
    953/954 MB (99%)
    954/954 MB (99%)
    954/954 MB (100%)
[*] Aciliyor -> /content/qairt/2.28.0.241029
[*] 180 dosyaya calistirma izni verildi (bin/)
[+] QNN_SDK_ROOT = /content/qairt/2.28.0.241029/qairt/2.28.0.241029
QNN_SDK_ROOT=/content/qairt/2.28.0.241029/qairt/2.28.0.241029

QNN_SDK_ROOT = /content/qairt/2.28.0.241029/qairt/2.28.0.241029


## 4) Modeli indir

Resmi hat `.safetensors` dosyasını **doğrudan** kullanır. İndirme mantığı
`scripts/fetch_ckpt.py` içinde: token ekleme, yarım dosyadan devam, HTML/eksik
dosya kontrolü.

> **civitai** indirme uçları artık **token** ister (tokensiz `401`).
> civitai.com → **Account settings → API Keys → Add API key**, sonra Colab
> **🔑 Secrets → `CIVITAI_TOKEN`** (*Notebook access* açık). Gated **HF** linki
> için aynı şekilde `HF_TOKEN`.


In [ ]:
%cd /content/sd-qnn
import os
try:
    from google.colab import userdata
    for k in ("CIVITAI_TOKEN", "HF_TOKEN"):
        v = userdata.get(k)
        if v: os.environ[k] = v
except Exception as e:
    print("[!] Secrets:", e)
!python scripts/fetch_ckpt.py --out work/input.safetensors


## 5) Dönüştür

Aşamalar: `uv` ortamı → `prepare_data` → `gen_quant_data` → `export_onnx` →
`qnn-onnx-converter` → `qnn-model-lib-generator` → `qnn-context-binary-generator`

> **Sekmeyi ön planda tutun.** Mobilde başka uygulamaya geçince tarayıcı sekmeyi
> askıya alıyor ve Colab çalışma zamanı kapanıyor. `CACHE_REPO` doluysa en pahalı
> adım (`prepare_data`, ~35 dk) HF'e yedeklenir; kopan oturum sonrası aynı
> hücreyi çalıştırmak o adımı **atlar**.

`CALIB_LIMIT` yalnızca **kuantizasyon** süresini etkiler — `prepare_data`
her hâlükârda tüm promptları üretir, o adım sabit maliyettir.


In [ ]:
#@title Dönüştür { display-mode: "form" }
#@markdown **Onbellek deposu** (HF, ozel dataset): prepare_data ciktisi
#@markdown (`data.pkl` + `images/`) buraya yedeklenir. Calisma zamani kapanirsa
#@markdown yeni oturum indirip o ~35 dk'lik adimi ATLAR. Bos = kapali.
CACHE_REPO = "sd-qnn-cache"  #@param {type:"string"}

%cd /content/sd-qnn
import os
assert os.environ.get("QNN_SDK_ROOT"), "Once 3. hucreyi calistirin"
try:
    from google.colab import userdata
    t = userdata.get("HF_TOKEN")
    if t: os.environ["HF_TOKEN"] = t
except Exception as e:
    print("[!] Secrets:", e)
os.environ["CACHE_REPO"] = CACHE_REPO.strip()
if CACHE_REPO.strip() and not os.environ.get("HF_TOKEN"):
    print("[!] Onbellek icin HF_TOKEN gerekli (Secrets) — onbellek kapali")
name = os.environ["MODEL_NAME"]
!bash scripts/06_official_pipeline.sh work/input.safetensors "{name}" "work/{name}" "$SOC"


## 6) Paketi indir

`BITTI` satırını gördükten sonra çalıştırın. **Hemen indirin** — çalışma zamanı
kapanırsa dosya kaybolur.

In [ ]:
import glob, os
from google.colab import files
zips = sorted(glob.glob("/content/sd-qnn/dist/*.zip"), key=os.path.getmtime)
assert zips, "dist/ bos — donusum tamamlanmadi"
z = zips[-1]
print(f"{z}  ({os.path.getsize(z)/1e6:.0f} MB)")
!unzip -l "{z}"
files.download(z)


/content/sd-qnn/dist/DeliberateCyber_qnn2.28_min.zip  (995 MB)
Archive:  /content/sd-qnn/dist/DeliberateCyber_qnn2.28_min.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   236544  2026-07-28 23:29   pos_emb.bin
 75890688  2026-07-28 23:29   token_emb.bin
 96453504  2026-07-28 23:55   vae_decoder.bin
892521824  2026-07-29 00:08   unet.bin
 58809328  2026-07-28 23:36   vae_encoder.bin
  3642034  2026-07-28 23:28   tokenizer.json
156316304  2026-07-28 23:29   clip_v2.mnn
---------                     -------
1283870226                     7 files


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7) Hugging Face'e yükle (isteğe bağlı)

Telefona indirmek yerine (ya da ek olarak) ZIP'i HF'e koyar: çalışma zamanı
kapansa bile kalıcı olur ve telefondan doğrudan indirilebilir.

**Gerekli:** yazma (write) izinli token → Colab sol menü **🔑 Secrets →
`HF_TOKEN`** (*Notebook access* açık olmalı).

Alanlar bu hücrenin kendi formunda — başka hücreye bağımlı değil.


In [ ]:
#@title Hugging Face'e yükle { display-mode: "form" }
#@markdown **Ayri repo:** her model `<kullanici>/<MODEL_NAME>` reposuna gider.
#@markdown **Koleksiyon:** hepsi tek repoda, her model kendi alt klasorunde.
UPLOAD_MODE = "Ayri repo"  #@param ["Ayri repo", "Koleksiyon"]
#@markdown Koleksiyon modunda kullanilacak repo adi
COLLECTION_REPO = "sd_qnn"  #@param {type:"string"}
#@markdown Elle tam repo adi (`kullanici/repo`) — doluysa yukaridakiler yok sayilir
HF_REPO = ""  #@param {type:"string"}
HF_PRIVATE = False  #@param {type:"boolean"}

import glob, os, subprocess, sys
os.chdir("/content/sd-qnn")
assert os.path.isdir("scripts"), "Once 2. adimi (depoyu cek) calistirin!"

zips = sorted(glob.glob("dist/*.zip"), key=os.path.getmtime)
assert zips, "dist/ icinde zip yok — 5. adim (donusum) tamamlanmamis olabilir."
zip_path = zips[-1]
print(f"{zip_path}  ({os.path.getsize(zip_path)/1e6:.0f} MB)")

# Token: Colab Secrets -> HF_TOKEN (write izinli olmali)
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok
except Exception as e:
    print("[!] Secrets okunamadi:", e)
assert os.environ.get("HF_TOKEN"), \
    "HF_TOKEN yok — Colab Secrets (anahtar simgesi) -> HF_TOKEN ekleyin (write)."

try:
    import huggingface_hub  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "huggingface_hub"], check=True)

# 1. hucre calistirilmadiysa model adini zip isminden turet
name = os.environ.get("MODEL_NAME", "").strip() or \
    os.path.basename(zip_path).split("_qnn")[0]

cmd = [sys.executable, "scripts/upload_hf.py", "--file", zip_path,
       "--name", name]
if HF_REPO.strip():
    cmd += ["--repo", HF_REPO.strip()]
elif UPLOAD_MODE == "Koleksiyon":
    cmd += ["--collection", COLLECTION_REPO.strip()]
if HF_PRIVATE:
    cmd.append("--private")

print(">", " ".join(cmd))
rc = subprocess.run(cmd).returncode
assert rc == 0, f"Yukleme basarisiz (cikis kodu {rc}) — yukaridaki hataya bakin."
